<a href="https://colab.research.google.com/github/thakurakanksha288/AI-ASSIGNMENT-PYTHON-/blob/main/assignment8_1024160019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Neural-Watch: Deep Learning for Sensor Anomaly Detection
### *An Unsupervised Approach to Predictive Maintenance*

**Objective:** To detect faults in industrial sensors without using labeled "error" data.
**Methodology:** Using an **Autoencoder Neural Network** to learn the "latent representation" of normal operations and flagging deviations as potential failures.

---
## 1. The Strategy
In the real world, anomalies are rare and often unlabeled. We use an **Unsupervised Learning** approach:
1. **Train** the model only on "Normal" operational data.
2. **Predict** on new data.
3. **Analyze Reconstruction Error:** If the model cannot "re-draw" the data accurately, it's a fault.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Creating a multi-sensor dataset with injected faults
t = np.linspace(0, 100, 2000)
temp = np.sin(t/5) + np.random.normal(0, 0.05, 2000)
vibe = np.random.normal(0.5, 0.1, 2000)

# Injecting a 'hidden' fault (a subtle drift in vibration)
vibe[1500:] += np.linspace(0, 1.2, 500)

df = pd.DataFrame({'Time': t, 'Temp': temp, 'Vibration': vibe})

In [ ]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Temp', 'Vibration']])

def create_sequences(data, window=20):
    return np.array([data[i:i+window] for i in range(len(data)-window)])

X_seq = create_sequences(scaled_data)
X_flat = X_seq.reshape(X_seq.shape[0], -1)

In [ ]:
inputs = Input(shape=(X_flat.shape[1],))
# Encoder: Compressing data
encoded = Dense(32, activation='relu')(inputs)
latent = Dense(4, activation='relu')(encoded) # The Bottleneck
# Decoder: Reconstructing data
decoded = Dense(32, activation='relu')(latent)
outputs = Dense(X_flat.shape[1], activation='linear')(decoded)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse')
model.fit(X_flat, X_flat, epochs=50, batch_size=32, verbose=0)

In [ ]:
preds = model.predict(X_flat)
mae_loss = np.mean(np.abs(preds - X_flat), axis=1)

# Statistical Threshold: Anything 3 standard deviations away is a 'Fault'
threshold = np.mean(mae_loss) + (3 * np.std(mae_loss))
df['Score'] = np.concatenate([np.zeros(20), mae_loss])
df['Anomaly'] = df['Score'] > threshold

62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Time'], y=df['Vibration'], name='Vibration Reading'))
# Overlaying red markers only where an anomaly is detected
anoms = df[df['Anomaly']]
fig.add_trace(go.Scatter(x=anoms['Time'], y=anoms['Vibration'], mode='markers',
                         marker=dict(color='red'), name='AI Detected Fault'))

fig.update_layout(template='plotly_dark', title="Neural-Watch: Live Fault Diagnostics")
fig.show()

The graph shows vibration readings staying fairly stable between about 0.4 and 0.7 for most of the timeline, indicating the system is operating normally. A few early red points mark minor anomalies detected by the AI, but nothing too serious at that stage. However, after around the 75 mark, the vibration levels start rising sharply, signaling the onset of system degradation. By the end, the values increase significantly, reflecting clear instability and a rapidly developing fault—basically, it goes from calm to subtly concerning, and then to obviously problematic.

Neural-Watch is an unsupervised AI system designed for Predictive Maintenance in industrial settings. Unlike traditional alerts that rely on fixed limits, this project utilizes a Deep Learning Autoencoder to learn the "normal" behavioral rhythm of multiple sensors (Temperature, Vibration, Pressure) simultaneously.

The core innovation lies in the Bottleneck Architecture: by forcing data through a compressed latent space, the model learns to reconstruct healthy patterns perfectly. When a fault or drift occurs, the reconstruction error spikes, triggering an adaptive alarm. This creates a zero-label solution that identifies impending failures without needing a history of past breakdowns, visualized through a professional real-time dashboard.